# NOM / IR / PE — Domain Expansion: Trichotomous
3 agents, 4 items, ordinal preferences, ε(3)=0

In [2]:
# ── Colab セットアップ（最初に1回だけ実行）──────────────────────────
!git clone https://github.com/shiro-seminar/NOM-matching.git

import sys
sys.path.insert(0, "/content/NOM-matching")

Cloning into 'NOM-matching'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 99 (delta 41), reused 76 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 86.19 KiB | 5.39 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [3]:
# コードを更新したいときはここを実行
!git -C /content/NOM-matching pull

Already up to date.


In [4]:
import torch
from domain_expansion_experiments.config      import Config
from domain_expansion_experiments.domains     import DOMAINS
from domain_expansion_experiments.train       import train
from domain_expansion_experiments.evaluate    import evaluate_mechanism, print_table
from domain_expansion_experiments.benchmarks  import BENCHMARKS
from domain_expansion_experiments.data_gen    import sample_batch
from domain_expansion_experiments.model       import AllocationNet
from domain_expansion_experiments.allocations import ir_pe_mask

## 1. Config

In [10]:
DOMAIN = "trichotomous_extended_e3"   # trichotomous / trichotomous_extended_e3 / four_chotomous_e4 / strict

cfg = Config(
    domain     = DOMAIN,
    steps      = 50_000,
    S          = 8,
    M          = 8,
    batch_size = 64,
    device     = "cuda" if torch.cuda.is_available() else "cpu",
    seed       = 42,
)
print(cfg)
print(f"device: {cfg.device}  num_ranks: {cfg.num_ranks}")

Config(num_agents=3, num_items=4, domain='trichotomous_extended_e3', num_ranks=4, hidden=256, depth=4, batch_size=64, steps=50000, lr=0.0003, grad_clip=1.0, seed=42, S=8, M=8, temperature=1.0, lambda_nom=0.0, rho=1.0, rho_mult=1.005, rho_max=200.0, dual_update_every=100, nom_target=0.005, welfare_weight=0.02, device='cuda')
device: cuda  num_ranks: 4


## 2. Train

In [ ]:
net = train(cfg, verbose=True)
net.eval()

  step=     1  welfare=-2.9959  nom=0.00000  lambda=0.000  rho=1.00  elapsed=0s
  step=   200  welfare=-3.0469  nom=0.00000  lambda=0.000  rho=1.00  elapsed=12s
  step=   400  welfare=-3.1258  nom=0.00000  lambda=0.000  rho=1.00  elapsed=24s
  step=   600  welfare=-2.9830  nom=0.00000  lambda=0.000  rho=1.00  elapsed=35s
  step=   800  welfare=-2.8896  nom=0.00000  lambda=0.000  rho=1.00  elapsed=47s
  step=  1000  welfare=-2.5781  nom=0.00000  lambda=0.000  rho=1.00  elapsed=59s
  step=  1200  welfare=-3.0000  nom=0.00000  lambda=0.000  rho=1.00  elapsed=71s
  step=  1400  welfare=-3.1562  nom=0.00000  lambda=0.000  rho=1.00  elapsed=83s
  step=  1600  welfare=-2.9219  nom=0.00000  lambda=0.000  rho=1.00  elapsed=95s
  step=  1800  welfare=-2.8750  nom=0.00000  lambda=0.000  rho=1.00  elapsed=106s
  step=  2000  welfare=-3.2378  nom=0.00000  lambda=0.000  rho=1.00  elapsed=118s
  step=  2200  welfare=-2.7463  nom=0.00000  lambda=0.000  rho=1.00  elapsed=130s
  step=  2400  welfare=-3.

## 3. Evaluate — benchmarks vs learned net

In [7]:
EVAL_N  = 500    # evaluation sample size
EVAL_S  = 64     # NOM opponent samples
EVAL_M  = 64     # NOM misreport samples

eval_cfg = Config(domain=DOMAIN, batch_size=EVAL_N, device=cfg.device, seed=0)
torch.manual_seed(0)

domain = DOMAINS[DOMAIN]
batch  = sample_batch(eval_cfg)
marginal_rank = batch["marginal_rank"]
endow_idx     = batch["endow_idx"]
S             = batch["S"]

# WMAX: oracle upper bound on welfare (IR+PE constraint)
irpe_m = ir_pe_mask(eval_cfg, S, endow_idx)
wmax_s = (S.sum(1) + (1 - irpe_m) * (-1e9)).max(1).values

results = []
for bname, bfn in BENCHMARKS.items():
    print(f"Evaluating {bname}...")
    r = evaluate_mechanism(bname, bfn, eval_cfg, domain,
                           marginal_rank, endow_idx, S, wmax_s,
                           eval_S=EVAL_S, eval_M=EVAL_M)
    results.append(r)

def net_mech(cfg_, mr_, ei_, S_):
    mask_ = ir_pe_mask(cfg_, S_, ei_)
    return net(mr_, mask=mask_, temperature=1e-3)

results.append(evaluate_mechanism("LearnedNet", net_mech, eval_cfg, domain,
                                  marginal_rank, endow_idx, S, wmax_s,
                                  eval_S=EVAL_S, eval_M=EVAL_M))

print_table(results)

Evaluating Endowment...
Evaluating WMAX-IR...
Evaluating WMAX-PE...
Evaluating WMAX-IR-PE...
Evaluating TTC-ordinal...
Evaluating Random-IR-PE...

-------------------------------------------------------------------------------------
Mechanism             Welfare  W/WMAX  IR-viol%     PE%   IR+PE%  NOM-mean  NOM-viol%
-------------------------------------------------------------------------------------
Endowment             -1.9780   1.427       0.0    26.0     57.6   0.00000        0.0
WMAX-IR               -0.8860   0.639       0.0   100.0     17.8   0.00000        0.0
WMAX-PE               -0.8860   0.639      22.0   100.0     11.0   0.00000        0.0
WMAX-IR-PE            -1.3860   1.000       0.0    68.4    100.0   0.00000        0.0
TTC-ordinal           -1.0140   0.732       0.0    88.4     25.0   0.00000        0.0
Random-IR-PE          -1.3860   1.000       0.0    68.4    100.0   0.00000        0.0
LearnedNet            -1.3860   1.000       0.0    68.4    100.0   0.00000     

## 4. (Optional) チェックポイントを保存 / 読み込み

In [8]:
# Google Drive にモデルを保存したい場合
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copy(f"{DOMAIN}_net.pt", f"/content/drive/MyDrive/{DOMAIN}_net.pt")
print(f"saved: {DOMAIN}_net.pt")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'trichotomous_net.pt'

In [ ]:
# Drive から読み込む場合
CKPT = f"/content/drive/MyDrive/{DOMAIN}_net.pt"
ckpt = torch.load(CKPT, map_location=cfg.device)
net2 = AllocationNet(Config(**ckpt["cfg"]))
net2.load_state_dict(ckpt["state_dict"])
net2.eval()
print(f"loaded: step={ckpt.get('step', 'final')}")